# End-to-End Evaluation — results & stage-by-stage report cards

Reads the JSON reports written by `scripts/evaluate_all.py`, then rebuilds the
**pair-level detail** behind the focused one so every stage of the pipeline gets
its own classification report and confusion matrix.

§1–§2 work offline from `data/evaluations/`. **§3 recomputes from the run's
artifacts and the label file**, so it must run where the data lives (the VM) and
needs the focused report's run to still have its Parquet artifacts on disk.

Section 3 walks the pipeline in order — blocking → deterministic rules → the
Stage-4.25 gate → the Stage-4.5 matcher → clustering — and for each stage shows
how many pairs it got right and how many it got wrong.

**Two things to keep in mind while reading (see `docs/End-to-End-Evaluation-Guide.md`):**

- **Leakage.** The Stage-4.25 gate and the Stage-4.5 matcher were both trained on
  the gold labels. Read their sections only from a `strict` holdout report.
- **Each stage is scored on the pool it actually saw.** The gate never sees pairs
  the rules already auto-merged; the matcher only sees gate survivors. The
  two-class matrices are restricted accordingly; the three-class routing
  matrices are not — they cover every labeled pair, so the four of them are
  directly comparable and you can watch the population settle stage by stage.

## 0. Setup

In [ ]:
import subprocess
import sys
from pathlib import Path


def _service_root(start: Path) -> Path:
    for d in [start, *start.parents]:
        if (d / "data").is_dir() and (d / "src").is_dir():
            return d
    raise FileNotFoundError("could not locate empi-service/ (no ancestor has data/ + src/)")


SERVICE_ROOT = _service_root(Path.cwd())
if str(SERVICE_ROOT) not in sys.path:
    sys.path.insert(0, str(SERVICE_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.config import settings
from src.evaluation import stage_diagnostics as sd
from src.evaluation.holdout import DEFAULT_GOLD_LABELS
from src.evaluation.report_io import (
    flow_frame,
    load_reports,
    loss_frame,
    summary_frame,
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)

print("service root:", SERVICE_ROOT)
print("runs dir    :", settings.runs_dir)

### Chart styling

Three categorical hues from the validated palette, assigned to series in a fixed
sorted order so a series keeps its colour when other series appear or disappear.
Every chart is followed by its table view.

In [25]:
SERIES_COLORS = ["#2a78d6", "#eb6834", "#1baf7a"]  # blue, orange, aqua
INK, MUTED, GRID_C = "#0b0b0b", "#52514e", "#e6e5e1"


def color_map(series_names) -> dict:
    """Stable series -> colour. Keyed on the series name, never on its rank in
    the current chart, so filtering one series out never repaints the others."""
    return {name: SERIES_COLORS[i % len(SERIES_COLORS)]
            for i, name in enumerate(sorted(series_names))}


plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 9,
    "axes.grid": True,
    "axes.axisbelow": True,
    "axes.edgecolor": GRID_C,
    "axes.labelcolor": MUTED,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.color": GRID_C,
    "grid.linewidth": 0.6,
    "legend.frameon": False,
    "text.color": INK,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
})

## 1. (Optional) Run evaluations to add a new point

Skip this if you only want to read existing results.

`scripts/evaluate_all.py` does the whole thing in one session: pipeline over the
real data → score vs gold at **both** holdouts → pipeline over synthetic → score
vs entity truth. Gold gets both holdouts on purpose — `none` gives the tighter
clustering headline (8x more labeled positives), `strict` gives the honest
gate/matcher numbers.

Set `REUSE_REAL_RUN` to an existing `run_id` to skip re-running the real-data
pipeline, which is the slow part.

In [26]:
RUN_EVALUATIONS = False   # flip to True to add a new session
SESSION_ID = None          # e.g. "gate_v2_baseline"; None -> UTC timestamp
REUSE_REAL_RUN = None      # an existing run_id, to skip the slow real-data run

if RUN_EVALUATIONS:
    cmd = [sys.executable, "scripts/evaluate_all.py"]
    if SESSION_ID:
        cmd += ["--session-id", SESSION_ID]
    if REUSE_REAL_RUN:
        cmd += ["--reuse-real-run", REUSE_REAL_RUN]
    print("$", " ".join(str(c) for c in cmd), "\n")
    # Streamed rather than captured: the real-data pipeline run takes minutes
    # and a silent cell that long is indistinguishable from a hung one.
    proc = subprocess.run(cmd, cwd=SERVICE_ROOT)
    if proc.returncode:
        raise RuntimeError("evaluate_all.py failed — see the output above.")
else:
    print("Skipped — reading stored reports only.")

$ c:\Users\MiguelGarcia\.conda\envs\capstone_env\python.exe scripts/evaluate_all.py 



## 2. What results exist

In [27]:
reports = load_reports()
print(f"{len(reports)} stored report(s)\n")

summary = summary_frame(reports)
summary

12 stored report(s)



,session_id,evaluated_utc,run_id,source,holdout,labeled_pairs,positives,precision,recall,f1,TP,FP,FN,git_sha
0,20260731T190736Z,2026-07-31T19:11:25+00:00,20260731T190736Z_synthetic,synthetic,n/a,10000,2000,0.9801,0.8380,0.9035,1676,34,324,1b174d04
1,20260731T190736Z,2026-07-31T19:11:14+00:00,20260731T190736Z_real,gold,strict,40849,12419,0.9901,0.5559,0.7120,6904,69,5515,1b174d04
2,20260731T190736Z,2026-07-31T19:11:10+00:00,20260731T190736Z_real,gold,none,204805,62537,0.9910,0.5503,0.7076,34414,314,28123,1b174d04
3,20260731T190206Z,2026-07-31T19:05:52+00:00,20260731T190206Z_synthetic,synthetic,n/a,10000,2000,0.9317,0.8870,0.9088,1774,130,226,1b174d04
4,20260731T190206Z,2026-07-31T19:05:41+00:00,20260731T190206Z_real,gold,strict,40849,12419,0.9579,0.6673,0.7866,8287,364,4132,1b174d04
5,20260731T190206Z,2026-07-31T19:05:37+00:00,20260731T190206Z_real,gold,none,204805,62537,0.9599,0.6636,0.7847,41497,1735,21040,1b174d04
6,20260731T173807Z,2026-07-31T17:41:52+00:00,20260731T173807Z_synthetic,synthetic,n/a,10000,2000,0.9429,0.8835,0.9122,1767,107,233,bd5357ea
7,20260731T173807Z,2026-07-31T17:41:41+00:00,20260731T173807Z_real,gold,strict,30936,2495,0.8298,0.6505,0.7293,1623,333,872,bd5357ea
8,20260731T173807Z,2026-07-31T17:41:37+00:00,20260731T173807Z_real,gold,none,204805,62537,0.9623,0.6577,0.7814,41130,1612,21407,bd5357ea
9,20260729T163951Z,2026-07-29T16:43:42+00:00,20260729T163951Z_synthetic,synthetic,n/a,10000,2000,0.7882,0.8765,0.8300,1753,471,247,85f447b3


## 3. One report in detail — a report card per stage

`FOCUS = 0` is the most recently evaluated report; change it to inspect another
row of the table above. `LABELS` must point at the label file that report was
scored against — the gold file by default.

Everything below is rebuilt from the run's own artifacts under exactly the
report's holdout, so these matrices describe the same population the stored
report's counts do.

In [ ]:
FOCUS = 0
LABELS = DEFAULT_GOLD_LABELS   # point at the synthetic/silver file for those reports

reports = load_reports()
report = reports[FOCUS]

diag = sd.diagnostics_for_report(report, labels=LABELS, settings=settings)

print(f"run {report['run_id']}  |  {report['label_source']}  |  "
      f"holdout: {report['leakage']['restriction']}")
print(f"evaluated {report['evaluated_utc']}  |  pipeline git sha: {report['git_sha']}")
print(diag.header())
if report["leakage"].get("note"):
    print("\n!", report["leakage"]["note"])

### 3.0 How to read the two views

Every stage below gets one or both of these.

**The two-class view** — the stage's own decision, on the pairs it actually saw,
against the target it was built to hit. The gate is scored against *plausible*
(match ∪ ambiguous), because keeping an ambiguous pair alive is correct behavior
for a filter; the matcher against *confident match*, because merging an
ambiguous pair is not what it was trained to do. Scoring either against the bare
match label reports a failure that is actually the design.

**The three-class routing view** — where each pair *should* have gone
(non-match → `no_match`, ambiguous → `human_review`, match → `auto_merge`)
versus where the pipeline had it once that stage had spoken. This one runs over
**every** labeled pair, so the diagonal is correct routing, above the diagonal is
too aggressive, and below it is too cautious.

The heatmap shades by row share (per-class recall) and prints the absolute count
in each cell — non-matches outnumber matches several to one, so a count-shaded
grid would be one dark corner and two invisible rows.

In [ ]:
def plot_matrix(counts: pd.DataFrame, pct: pd.DataFrame, *, title: str,
                xlabel: str = "system decision", ylabel: str = "label"):
    """Confusion matrix -> annotated heatmap. One sequential hue, light -> dark,
    per the palette's magnitude rule; cell text flips to white on the dark end
    so every label stays legible."""
    counts = counts.drop(columns="total", errors="ignore")
    fig, ax = plt.subplots(figsize=(1.55 * len(counts.columns) + 2.6,
                                    0.75 * len(counts.index) + 1.9))
    ax.imshow(pct.to_numpy(dtype=float), cmap="Blues", vmin=0, vmax=100)

    ax.set_xticks(range(len(counts.columns)), counts.columns)
    ax.set_yticks(range(len(counts.index)), counts.index)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title, color=INK, loc="left")
    ax.grid(visible=False)
    for spine in ax.spines.values():
        spine.set_visible(False)

    for r in range(counts.shape[0]):
        for c in range(counts.shape[1]):
            share = pct.iat[r, c]
            share = 0.0 if pd.isna(share) else float(share)
            ax.text(c, r, f"{int(counts.iat[r, c]):,}\n{share:.1f}%",
                    ha="center", va="center", fontsize=8.5,
                    color="white" if share > 55 else INK)
    plt.tight_layout()
    plt.show()


def binary_view(stage: str, title: str | None = None):
    """Confusion matrix + classification report + heatmap for one stage."""
    spec = sd.BINARY_STAGES[stage]
    cm = sd.binary_confusion(diag, stage)
    pct = sd.binary_confusion(diag, stage, normalize=True)
    display(cm)
    display(sd.binary_report(diag, stage))
    plot_matrix(cm, pct, title=title or spec.title,
                xlabel="stage decision", ylabel="label")


def routing_view(stage: str, title: str | None = None, population: str | None = None):
    """Three-class routing matrix + classification report + heatmap."""
    cm = sd.route_confusion(diag, stage, population=population)
    pct = sd.route_confusion(diag, stage, normalize=True, population=population)
    display(cm)
    display(sd.route_report(diag, stage, population=population))
    plot_matrix(cm, pct, title=title or f"Routing after {stage}",
                xlabel="system routed to", ylabel="expected route")

### 3.1 Blocking — was the pair ever considered?

Blocking is a pure filter, and the only cell that costs anything is
**true match → not blocked**: no later stage can recover a pair that was never
emitted. False positives are free by design — blocking over-generates and the
rules, the gate and the matcher exist to clean up after it, so a low precision
here is the intended shape, not a defect.

In [ ]:
binary_view("blocking")

In [ ]:
# The unrecoverable misses, by how much evidence the pair carried elsewhere.
missed = sd.binary_errors(diag, "blocking", kind="FN")
print(f"{len(missed):,} true pairs blocking never emitted "
      f"({len(missed) / max(int(diag.pairs['gold_match'].sum()), 1):.2%} of all true pairs)")

### 3.2 Deterministic rules — the three-way decision

The rules make all three calls at once: confirm and auto-merge, reject on
contradictions, or hand the pair on. So this stage is read as a three-class
matrix rather than a binary one.

The first matrix is **cumulative** — the pairs blocking never emitted are
already sitting in the `no_match` column, because that is where the pipeline has
in fact left them. The second restricts to the blocked pool, which isolates what
the rules themselves decided.

Two cells to watch: `match → no_match` (a true pair the reject rules threw away,
unrecoverable) and `non-match → auto_merge` (a wrong merge that ships, since the
rules' auto-merge tier is what clustering unions).

In [ ]:
routing_view("rules", title="Routing after the deterministic rules")

In [ ]:
routing_view("rules", title="The rules' own decision (blocked pairs only)",
             population="blocked")

### 3.3 Stage-4.25 gate — the confident non-match filter

The gate scores only the rules' `non_matches` pool, and it makes one binary
call: **plausible** (pass it on) or **confident non-match** (drop it). Its drops
are unrecoverable, so recall on `plausible` is the number that matters —
precision is cheap here, because everything it passes is re-examined downstream.

In [ ]:
binary_view("gate")

#### Routing after the gate

Same three classes, now over every labeled pair. Compare it cell for cell with
§3.2: the only movement is `human_review → no_match`, since a filter can close
a pair but never merge one.

In [ ]:
routing_view("gate", title="Routing after the Stage-4.25 gate")

### 3.4 Stage-4.5 matcher — the confident match classifier

The matcher sees only the gate's survivors and decides **confident match**
(`auto_merge`) or **ambiguous** (leave it in review). It has no `no_match` tier —
by construction it cannot drop a pair.

Its target is *confident match* (match and not ambiguous), so auto-merging a
pair the labeler could not resolve counts against it here even if the pair is
in fact a match. With `ml_feeds_clustering` on, that false-positive cell is a
real wrong merge in the shipped output.

In [ ]:
binary_view("ml_matcher")

#### Routing after the matcher

The first column where pairs move *toward* `auto_merge`. Everything still in
`human_review` at this point is the reviewer queue.

In [ ]:
routing_view("ml_matcher", title="Routing after the Stage-4.5 matcher")

### 3.5 Clustering — the shipped decision

Clustering takes the connected components of the binding `auto_merge` edges, so
it can merge a pair **no classifier ever scored** — transitive closure. Those
merges are invisible to every matrix above, which is why the final numbers can
be worse than the inputs that produced them.

The two-class view is the headline: did we merge this pair or not, against the
match label.

In [ ]:
binary_view("clustering")

#### Final routing — the decision the pipeline actually makes

This reproduces the stored report's triage matrix. It is the fair recall view:
an ambiguous pair sitting in `human_review` is scored as **correct**, because
routing undecidable evidence to a human is exactly what the pipeline is for.

`auto_merge` recall is the share of confident matches resolved without a human.
`human_review` recall says whether genuinely undecidable pairs are reaching a
reviewer rather than being silently dropped — a low value there is the serious
failure, because a pair dropped at `no_match` is unrecoverable.

In [ ]:
routing_view("clustering", title="Final routing — the shipped decision")

In [ ]:
# Merged only because of transitive closure: in the same cluster, but no stage
# ever emitted an auto_merge edge for the pair.
transitive = diag.pairs[diag.pairs["clustered"]
                        & (diag.pairs["rules_decision"] != "auto_merge")
                        & ~diag.pairs["ml_auto"]]
print(f"{len(transitive):,} labeled pairs merged by transitive closure alone — "
      f"{int((~transitive['gold_match']).sum()):,} of them labeled non-matches")

### 3.6 Where the true pairs were lost

The matrices above say how each stage did; this says which stage is responsible
for the misses. Each missed true pair is attributed to the **first** stage that
dropped it, so a pair the rules rejected is never also blamed on the gate.
Blocking and the gate are the two to watch — a pair either never blocked or
dropped by the gate is unrecoverable, while one left in review is merely slow.

These two tables come from the stored report, not from the recomputation above.

In [ ]:
flow_frame(report)[['stage', 'saw', 'auto_merge', 'no_match', 'to_next', 'true_lost']]

In [ ]:
losses = loss_frame(report)
losses

In [ ]:
def plot_losses(losses: pd.DataFrame, title: str):
    """Magnitude by category -> horizontal bars, sorted. One series, so no
    legend; every bar is directly labelled instead of relying on the axis."""
    data = losses[losses["missed"] > 0]
    if data.empty:
        print("No missed true pairs to attribute.")
        return
    fig, ax = plt.subplots(figsize=(7.2, 0.42 * len(data) + 1.4))
    y = range(len(data))
    ax.barh(list(y), data["missed"], color=SERIES_COLORS[0], height=0.62)
    ax.set_yticks(list(y), data["stage"])
    ax.invert_yaxis()
    ax.set_xlabel("true pairs lost")
    ax.set_title(title, color=INK, loc="left")
    ax.grid(axis="y", visible=False)
    span = data["missed"].max()
    for i, (n, pct) in enumerate(zip(data["missed"], data["share_pct"])):
        ax.text(n + span * 0.015, i, f"{n:,}  ({pct}%)", va="center",
                fontsize=8.5, color=MUTED)
    ax.set_xlim(0, span * 1.25)
    plt.tight_layout()
    plt.show()


plot_losses(losses, f"Where true pairs were lost — {report['label_source']} / "
                    f"{report['leakage']['restriction']}")

---

**The pairs behind these cells** live in `misclassified_pairs.ipynb`, which
follows the same section order and lists the actual records for every error cell
above. It is kept separate because its output is PHI.